# 01 - Extract and Profile Source Data

Generate realistic operational source data and profile the raw CSV files. Run this notebook first.

## Setup Project Paths

Prepare folder references for raw data, processed data, and outputs. The path logic works whether the notebook is opened from the project root or from inside `notebooks/`.

**Input:** current working directory.  
**Output:** `RAW_DIR`, `PROCESSED_DIR`, `OUTPUT_DIR`.

In [1]:
from pathlib import Path
from datetime import date, datetime, timedelta
import random
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'output'
for folder in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

D:\Semester 8\Data Warehouse\erajaya-data-warehouse


## Define Source Data Rules

Define the business values used by the dummy data generator: brands, categories, store types, channels, payment methods, loyalty tiers, and Indonesian locations.

**Output:** constants and helper functions for data generation.

In [2]:
SEED = 42
BRANDS = ['Apple', 'Samsung', 'Xiaomi', 'OPPO', 'vivo', 'ASUS', 'JBL', 'Anker', 'Logitech']
CATEGORIES = ['Smartphone', 'Tablet', 'Laptop', 'Accessories', 'Wearable', 'Audio', 'Smart Home']
STORE_TYPES = ['iBox', 'Erafone', 'Urban Republic', 'Eraspace']
CHANNELS = [{'channel_id':'CH_OFF','channel_name':'Offline Store','channel_type':'Offline'}, {'channel_id':'CH_WEB','channel_name':'Online Web','channel_type':'Online'}, {'channel_id':'CH_MKT','channel_name':'Marketplace','channel_type':'Online'}]
PAYMENT_METHODS = ['Cash', 'Debit Card', 'Credit Card', 'QRIS', 'Bank Transfer', 'E-Wallet']
LOYALTY_TIERS = ['Regular', 'Silver', 'Gold', 'Platinum']
PROVINCES = {'DKI Jakarta':['Jakarta Pusat','Jakarta Selatan','Jakarta Barat','Jakarta Utara'], 'Jawa Barat':['Bandung','Bekasi','Depok','Bogor'], 'Jawa Timur':['Surabaya','Malang','Sidoarjo'], 'Banten':['Tangerang','Tangerang Selatan'], 'Bali':['Denpasar','Badung'], 'Sumatera Utara':['Medan','Deli Serdang']}
FIRST_NAMES = ['Adi','Budi','Citra','Dewi','Eka','Fajar','Gita','Hana','Indra','Joko','Lina','Maya','Nadia','Putra','Rani','Sari','Tono','Vina','Wahyu','Yuni']
LAST_NAMES = ['Pratama','Saputra','Wijaya','Santoso','Lestari','Permana','Halim','Gunawan','Siregar','Susanto']

def rupiah_price(category, brand):
    ranges = {'Smartphone':(1_800_000,22_000_000),'Tablet':(2_500_000,18_000_000),'Laptop':(5_500_000,35_000_000),'Accessories':(80_000,2_500_000),'Wearable':(350_000,8_000_000),'Audio':(150_000,7_000_000),'Smart Home':(250_000,6_000_000)}
    low, high = ranges[category]
    premium = 1.25 if brand in {'Apple','Samsung','Asus','Lenovo'} else 1.0
    return int(round(random.randrange(low, high, 50_000) * premium, -3))

def make_customers(n=999):
    rows = [{'customer_id':'CUST-GUEST','customer_name':'Guest Customer','gender':'Unknown','birth_date':'','email':'','phone':'','loyalty_tier':'Regular','customer_segment':'Walk-in','province':'','city':'','registered_date':'2023-01-01'}]
    for i in range(1, n + 1):
        first, last = random.choice(FIRST_NAMES), random.choice(LAST_NAMES)
        province = random.choice(list(PROVINCES)); city = random.choice(PROVINCES[province])
        rows.append({'customer_id':f'CUST-{i:04d}','customer_name':f'{first} {last}','gender':random.choice(['Male','Female']),'birth_date':str(date(random.randint(1975,2005), random.randint(1,12), random.randint(1,28))),'email':f'{first.lower()}.{last.lower()}{i}@example.com','phone':f'08{random.randint(1000000000,9999999999)}','loyalty_tier':random.choices(LOYALTY_TIERS, weights=[48,28,18,6])[0],'customer_segment':random.choice(['Student','Professional','Family','Gadget Enthusiast','SMB']),'province':province,'city':city,'registered_date':str(date(2022,1,1)+timedelta(days=random.randint(0,1150)))})
    return pd.DataFrame(rows)

def make_products(n=100):
    catalog = [
        ('Apple iPhone 16 Pro Max 256GB Desert Titanium', 'Apple', 'Smartphone', 'Flagship', 20499000),
        ('Apple iPhone 16 Pro Max 512GB Natural Titanium', 'Apple', 'Smartphone', 'Flagship', 23499000),
        ('Apple iPhone 16 Pro 256GB Black Titanium', 'Apple', 'Smartphone', 'Flagship', 18499000),
        ('Apple iPhone 16 Pro 128GB White Titanium', 'Apple', 'Smartphone', 'Flagship', 16999000),
        ('Apple iPhone 16 Plus 128GB Pink', 'Apple', 'Smartphone', 'Premium', 13999000),
        ('Apple iPhone 16 128GB Ultramarine', 'Apple', 'Smartphone', 'Premium', 12499000),
        ('Apple iPhone 16e 128GB Black', 'Apple', 'Smartphone', 'Premium', 9999000),
        ('Apple iPhone 15 128GB Black', 'Apple', 'Smartphone', 'Premium', 10999000),
        ('Apple iPhone 15 Plus 128GB Blue', 'Apple', 'Smartphone', 'Premium', 12999000),
        ('Apple iPad Air 11-inch M4 Wi-Fi 128GB Space Grey', 'Apple', 'Tablet', 'Premium', 12599000),
        ('Apple iPad Air 13-inch M4 Wi-Fi 128GB Blue', 'Apple', 'Tablet', 'Premium', 15999000),
        ('Apple iPad Air 11-inch M3 Wi-Fi 128GB Space Grey', 'Apple', 'Tablet', 'Premium', 11499000),
        ('Apple iPad 10th Gen Wi-Fi 64GB Pink', 'Apple', 'Tablet', 'Midrange', 6499000),
        ('Apple iPad mini A17 Pro Wi-Fi 128GB Space Grey', 'Apple', 'Tablet', 'Premium', 9499000),
        ('Apple MacBook Air 13.6-inch M2 2024 16GB/256GB Midnight', 'Apple', 'Laptop', 'Premium', 13599000),
        ('Apple MacBook Air 13.6-inch M3 2024 16GB/256GB Midnight', 'Apple', 'Laptop', 'Premium', 14999000),
        ('Apple MacBook Air 15-inch M3 2024 16GB/512GB Starlight', 'Apple', 'Laptop', 'Premium', 22999000),
        ('Apple Watch Series 10 GPS 46mm Black', 'Apple', 'Wearable', 'Premium', 7999000),
        ('Apple Watch Ultra 2 GPS Cellular 49mm Natural Titanium', 'Apple', 'Wearable', 'Premium', 14999000),
        ('Apple AirPods Pro 2 USB-C', 'Apple', 'Audio', 'Premium', 3999000),
        ('Apple AirPods 4', 'Apple', 'Audio', 'Premium', 2499000),
        ('Apple 20W USB-C Power Adapter', 'Apple', 'Accessories', 'Charging', 499000),
        ('Apple Watch Magnetic Fast Charger to USB-C Cable 1m', 'Apple', 'Accessories', 'Charging', 499000),
        ('Apple 240W USB-C Charge Cable 2m', 'Apple', 'Accessories', 'Charging', 399000),
        ('Samsung Galaxy S26 Ultra 256GB Titanium Gray', 'Samsung', 'Smartphone', 'Flagship', 24499000),
        ('Samsung Galaxy S26 Plus 256GB Navy', 'Samsung', 'Smartphone', 'Flagship', 19499000),
        ('Samsung Galaxy S26 256GB Icy Blue', 'Samsung', 'Smartphone', 'Flagship', 16499000),
        ('Samsung Galaxy S25 Ultra 256GB Titanium Black', 'Samsung', 'Smartphone', 'Flagship', 20999000),
        ('Samsung Galaxy S25 Edge 256GB Titanium Silver', 'Samsung', 'Smartphone', 'Flagship', 17499000),
        ('Samsung Galaxy S25 256GB Navy', 'Samsung', 'Smartphone', 'Flagship', 13499000),
        ('Samsung Galaxy S24 Ultra 256GB Titanium Gray', 'Samsung', 'Smartphone', 'Flagship', 20999000),
        ('Samsung Galaxy Z Fold7 256GB Blue Shadow', 'Samsung', 'Smartphone', 'Foldable', 28499000),
        ('Samsung Galaxy Z Flip6 256GB Silver Shadow', 'Samsung', 'Smartphone', 'Foldable', 15999000),
        ('Samsung Galaxy A56 5G 8/256GB Awesome Graphite', 'Samsung', 'Smartphone', 'Midrange', 5999000),
        ('Samsung Galaxy A36 5G 8/256GB Awesome Lavender', 'Samsung', 'Smartphone', 'Midrange', 4999000),
        ('Samsung Galaxy A26 5G 8/256GB Black', 'Samsung', 'Smartphone', 'Midrange', 3999000),
        ('Samsung Galaxy A16 5G 8/256GB Blue Black', 'Samsung', 'Smartphone', 'Entry', 2999000),
        ('Samsung Galaxy A07 Special Hari Raya Package', 'Samsung', 'Smartphone', 'Entry', 1599000),
        ('Samsung Galaxy Tab S11 Wi-Fi 256GB Gray', 'Samsung', 'Tablet', 'Premium', 13999000),
        ('Samsung Galaxy Tab S10 Plus 5G 12/256GB Moonstone Gray', 'Samsung', 'Tablet', 'Premium', 15999000),
        ('Samsung Galaxy Tab S9 FE Wi-Fi 6/128GB Gray', 'Samsung', 'Tablet', 'Midrange', 6499000),
        ('Samsung Galaxy Tab A9 Plus 5G 8/128GB Graphite', 'Samsung', 'Tablet', 'Midrange', 3499000),
        ('Samsung Galaxy Watch Ultra LTE 47mm Titanium Gray', 'Samsung', 'Wearable', 'Premium', 8999000),
        ('Samsung Galaxy Watch7 44mm Green', 'Samsung', 'Wearable', 'Premium', 4499000),
        ('Samsung Galaxy Watch5 Pro 45mm Black', 'Samsung', 'Wearable', 'Premium', 3999000),
        ('Samsung Galaxy Buds3 Pro Silver', 'Samsung', 'Audio', 'Premium', 2999000),
        ('Xiaomi 15 12/512GB Green', 'Xiaomi', 'Smartphone', 'Flagship', 12499000),
        ('Xiaomi 15 Ultra 16/512GB Black', 'Xiaomi', 'Smartphone', 'Flagship', 16999000),
        ('Xiaomi 15T 12/256GB Black', 'Xiaomi', 'Smartphone', 'Flagship', 6999000),
        ('Xiaomi 14T Pro 12/512GB Titan Gray', 'Xiaomi', 'Smartphone', 'Flagship', 8499000),
        ('Xiaomi 14T 12/256GB Lemon Green', 'Xiaomi', 'Smartphone', 'Premium', 6499000),
        ('Xiaomi Redmi 15 8/128GB Midnight Black', 'Xiaomi', 'Smartphone', 'Entry', 2099000),
        ('Xiaomi Redmi Note 14 Pro Plus 5G 8/256GB Purple', 'Xiaomi', 'Smartphone', 'Midrange', 5099000),
        ('Xiaomi Redmi Note 14 Pro 5G 8/256GB Midnight Black', 'Xiaomi', 'Smartphone', 'Midrange', 4499000),
        ('Xiaomi Redmi Note 14 5G 8/256GB Coral Green', 'Xiaomi', 'Smartphone', 'Midrange', 3299000),
        ('Xiaomi Redmi Note 13 Pro Plus 5G 12/512GB Midnight Black', 'Xiaomi', 'Smartphone', 'Midrange', 5799000),
        ('Xiaomi POCO F6 12/512GB Black', 'Xiaomi', 'Smartphone', 'Premium', 5999000),
        ('Xiaomi POCO X6 Pro 5G 12/512GB Yellow', 'Xiaomi', 'Smartphone', 'Premium', 4999000),
        ('Xiaomi Redmi Pad 2 4/128GB Graphite Gray', 'Xiaomi', 'Tablet', 'Entry', 2299000),
        ('Xiaomi Pad 7 8/256GB Gray', 'Xiaomi', 'Tablet', 'Premium', 4999000),
        ('OPPO Reno14 F 5G 8/256GB Luminous Green', 'OPPO', 'Smartphone', 'Midrange', 5599000),
        ('OPPO Reno14 F 5G 12/256GB Glossy Pink', 'OPPO', 'Smartphone', 'Midrange', 5999000),
        ('OPPO Reno14 5G 12/256GB Opal Blue', 'OPPO', 'Smartphone', 'Premium', 6999000),
        ('OPPO Reno13 5G 12/256GB Plume White', 'OPPO', 'Smartphone', 'Premium', 7499000),
        ('OPPO Reno12 Pro 5G 12/512GB Nebula Silver', 'OPPO', 'Smartphone', 'Premium', 6299000),
        ('OPPO Reno12 5G 12/256GB Astro Silver', 'OPPO', 'Smartphone', 'Midrange', 5499000),
        ('OPPO A6 Pro 5G 8/256GB Stellar Blue', 'OPPO', 'Smartphone', 'Midrange', 3999000),
        ('OPPO A5 Pro 5G 8/256GB Mocha Brown', 'OPPO', 'Smartphone', 'Midrange', 3499000),
        ('OPPO A3x 4/64GB Sparkle Black', 'OPPO', 'Smartphone', 'Entry', 1599000),
        ('vivo V70 FE 5G 12/256GB Blue', 'vivo', 'Smartphone', 'Premium', 6499000),
        ('vivo V60 Lite 5G 8/256GB Pink', 'vivo', 'Smartphone', 'Midrange', 4999000),
        ('vivo V40 5G 12/512GB Chrome Purple', 'vivo', 'Smartphone', 'Premium', 6999000),
        ('vivo V40 Lite 5G 8/256GB Titanium Silver', 'vivo', 'Smartphone', 'Midrange', 4299000),
        ('vivo V30 5G 12/512GB Noble Black', 'vivo', 'Smartphone', 'Premium', 5999000),
        ('vivo V30e 5G 8/256GB Coco Brown', 'vivo', 'Smartphone', 'Midrange', 4699000),
        ('vivo Y29 8/256GB Diamond Black', 'vivo', 'Smartphone', 'Entry', 2999000),
        ('vivo Y19s 6/128GB Glossy Black', 'vivo', 'Smartphone', 'Entry', 1999000),
        ('ASUS ROG Phone 9 Pro 24GB/1TB Phantom Black', 'ASUS', 'Smartphone', 'Flagship', 21999000),
        ('ASUS ROG Phone 9 Pro 16GB/512GB Phantom Black', 'ASUS', 'Smartphone', 'Flagship', 18999000),
        ('ASUS ROG Phone 8 12/256GB Phantom Black', 'ASUS', 'Smartphone', 'Flagship', 10999000),
        ('ASUS ROG Phone 6 8/256GB White', 'ASUS', 'Smartphone', 'Premium', 7999000),
        ('ASUS Zenfone 10 8/128GB Eclipse Red', 'ASUS', 'Smartphone', 'Premium', 7499000),
        ('ASUS Zenfone 10 8/128GB Aurora Green', 'ASUS', 'Smartphone', 'Premium', 7499000),
        ('ASUS ROG 9 AeroActive Cooler X Pro', 'ASUS', 'Accessories', 'Gaming', 1399000),
        ('ASUS ROG 8 Antibacterial Glass Screen Protector', 'ASUS', 'Accessories', 'Protection', 399000),
        ('JBL Flip 6 White', 'JBL', 'Audio', 'Portable Speaker', 1199000),
        ('JBL Flip 6 Blue', 'JBL', 'Audio', 'Portable Speaker', 1199000),
        ('JBL Charge 6 Portable Speaker White', 'JBL', 'Audio', 'Portable Speaker', 2799000),
        ('JBL Clip 4 Red', 'JBL', 'Audio', 'Portable Speaker', 779000),
        ('JBL GO 4 Ultra-Portable Speaker White', 'JBL', 'Audio', 'Portable Speaker', 699000),
        ('JBL GO 4 Ultra-Portable Speaker Sand', 'JBL', 'Audio', 'Portable Speaker', 699000),
        ('JBL Tune 510BT Rose', 'JBL', 'Audio', 'Headphone', 649000),
        ('JBL T520 Bluetooth Headphone Black', 'JBL', 'Audio', 'Headphone', 699000),
        ('JBL Live Flex 3 Black', 'JBL', 'Audio', 'Earbuds', 2999000),
        ('Anker Soundcore Select 4 Go Black', 'Anker', 'Audio', 'Portable Speaker', 399000),
        ('Anker Soundcore Boom 2 SE Black', 'Anker', 'Audio', 'Portable Speaker', 1599000),
        ('Logitech MX Master 3S for Mac Pale Grey', 'Logitech', 'Accessories', 'Mouse', 1565000),
        ('Logitech MX Keys Mini for Mac Pale Grey', 'Logitech', 'Accessories', 'Keyboard', 1695000),
        ('Logitech Combo Touch iPad Air 11 M2 5th Gen Grey', 'Logitech', 'Accessories', 'Keyboard Case', 3079000),
        ('Xiaomi Google TV 32-inch A Pro HD', 'Xiaomi', 'Smart Home', 'Google TV', 1799000),
    ]
    if n > len(catalog):
        raise ValueError(f'Requested {n} products, but curated Erajaya catalog only contains {len(catalog)} products.')
    rows = []
    for i, (product_name, brand, category, subcategory, unit_price) in enumerate(catalog[:n], start=1):
        rows.append({
            'product_id': f'PROD-{i:04d}',
            'product_name': product_name,
            'brand': brand,
            'category': category,
            'subcategory': subcategory,
            'unit_price': unit_price,
            'cost_price': int(round(unit_price * 0.72, -3)),
            'launch_date': str(date(2023 + ((i - 1) % 3), ((i - 1) % 12) + 1, min(28, ((i * 3) % 28) + 1))),
            'is_active': True,
        })
    return pd.DataFrame(rows)
def make_stores(n=25):
    malls = ['Central Park','Grand Indonesia','Summarecon Mall','Tunjungan Plaza','Paris Van Java','Beachwalk','Galaxy Mall','Living World','Mall Kelapa Gading']
    rows = []
    for i in range(1, n + 1):
        store_type = STORE_TYPES[(i - 1) % len(STORE_TYPES)]; province = random.choice(list(PROVINCES)); city = random.choice(PROVINCES[province])
        rows.append({'store_id':f'STORE-{i:03d}','store_name':f'{store_type} {random.choice(malls)} {i}','store_type':store_type,'province':province,'city':city,'region':'Jabodetabek' if province in {'DKI Jakarta','Banten','Jawa Barat'} else 'Non-Jabodetabek','open_date':str(date(2018,1,1)+timedelta(days=random.randint(0,2200)))})
    return pd.DataFrame(rows)

def make_promotions():
    rows = [{'promotion_id':'PROMO-NONE','promotion_name':'No Promotion','promotion_type':'No Promotion','discount_rate':0.0,'start_date':'2024-01-01','end_date':'2026-12-31'}]
    promo_types = ['Cashback','Bundling','Bank Discount','Flash Sale','Loyalty Reward']
    for i in range(1, 21):
        start = date(2024,1,1) + timedelta(days=random.randint(0,650))
        rows.append({'promotion_id':f'PROMO-{i:03d}','promotion_name':f'{random.choice(promo_types)} Campaign {i}','promotion_type':random.choice(promo_types),'discount_rate':random.choice([0.03,0.05,0.08,0.10,0.12,0.15]),'start_date':str(start),'end_date':str(start + timedelta(days=random.randint(14,90)))})
    return pd.DataFrame(rows)

## Generate Operational Source Files

Create source-system CSV files for CRM, product master, store master, promotion, sales, payment, and inventory. These files intentionally represent operational data, not dim/fact tables yet.

**Output:** CSV files in `data/raw/`.

In [3]:
def make_transactions(customers, stores, promotions, products, n=2000):
    transaction_rows, detail_rows, payment_rows = [], [], []
    customer_ids = customers['customer_id'].tolist(); store_ids = stores['store_id'].tolist(); promos = promotions.to_dict('records')
    detail_id = 1; start_dt = datetime(2024, 1, 1, 9, 0)
    for i in range(1, n + 1):
        channel = random.choices(CHANNELS, weights=[62,23,15])[0]
        transaction_id = f'TRX-{i:05d}'; payment_id = f'PAY-{i:05d}'
        tx_date = start_dt + timedelta(days=random.randint(0,515), hours=random.randint(0,12), minutes=random.randint(0,59))
        transaction_rows.append({'transaction_id':transaction_id,'transaction_date':tx_date.strftime('%Y-%m-%d %H:%M:%S'),'customer_id':random.choice(customer_ids[1:]) if random.random() > 0.08 else '','store_id':random.choice(store_ids),'channel_id':channel['channel_id'],'channel_name':channel['channel_name'],'channel_type':channel['channel_type'],'payment_id':payment_id,'salesperson_id':f"EMP-{random.randint(1,80):03d}" if channel['channel_id'] == 'CH_OFF' else ''})
        tx_total = 0
        sampled_products = products.sample(n=random.choices([1,2,3,4], weights=[20,40,25,15])[0], replace=False, random_state=random.randint(1,999999))
        for _, product in sampled_products.iterrows():
            quantity = random.choices([1,2,3], weights=[78,17,5])[0]
            promo = random.choice(promos) if random.random() < 0.34 else promos[0]
            gross_sales = quantity * float(product['unit_price']); discount_amount = round(gross_sales * float(promo['discount_rate']), 2); tx_total += gross_sales - discount_amount
            detail_rows.append({'detail_id':f'DTL-{detail_id:06d}','transaction_id':transaction_id,'product_id':product['product_id'],'promotion_id':promo['promotion_id'],'quantity':quantity,'unit_price':product['unit_price'],'discount_amount':discount_amount})
            detail_id += 1
        payment_rows.append({'payment_id':payment_id,'payment_method':random.choices(PAYMENT_METHODS, weights=[10,15,22,20,15,18])[0],'payment_status':random.choices(['Paid','Pending','Failed','Refunded'], weights=[91,4,3,2])[0],'payment_provider':random.choice(['BCA','Mandiri','BRI','BNI','GoPay','OVO','ShopeePay','Visa','Mastercard','Erajaya POS']),'paid_amount':round(tx_total, 2)})
    return pd.DataFrame(transaction_rows), pd.DataFrame(detail_rows), pd.DataFrame(payment_rows)

def make_inventory(products, stores):
    rows = []
    for _, store in stores.iterrows():
        for _, product in products.iterrows():
            reorder_level = random.randint(5, 25)
            stock_quantity = random.choices([random.randint(0, reorder_level - 1), random.randint(reorder_level, 120)], weights=[18,82])[0]
            rows.append({'inventory_id':f"INV-{store['store_id']}-{product['product_id']}",'snapshot_date':'2025-05-31','store_id':store['store_id'],'product_id':product['product_id'],'stock_quantity':stock_quantity,'reorder_level':reorder_level,'stock_status':'Stockout' if stock_quantity == 0 else 'Low Stock' if stock_quantity < reorder_level else 'Available'})
    return pd.DataFrame(rows)

random.seed(SEED)
customers = make_customers(); products = make_products(); stores = make_stores(); promotions = make_promotions()
transactions, details, payments = make_transactions(customers, stores, promotions, products)
inventory = make_inventory(products, stores)
raw_frames = {'customers.csv':customers,'products.csv':products,'stores.csv':stores,'promotions.csv':promotions,'sales_transactions.csv':transactions,'sales_details.csv':details,'payments.csv':payments,'inventory.csv':inventory}
for filename, df in raw_frames.items():
    df.to_csv(RAW_DIR / filename, index=False)
    print(f'{filename}: {len(df):,} rows')

customers.csv: 1,000 rows
products.csv: 100 rows
stores.csv: 25 rows
promotions.csv: 21 rows
sales_transactions.csv: 2,000 rows
sales_details.csv: 4,688 rows
payments.csv: 2,000 rows
inventory.csv: 2,500 rows


## Profile Raw Data

Check each raw file before transformation: row count, column count, duplicate rows, and missing values.

**Input:** `data/raw/*.csv`.  
**Output:** raw data profile table.

In [4]:
profile = []
for path in sorted(RAW_DIR.glob('*.csv')):
    df = pd.read_csv(path)
    profile.append({'file': path.name, 'rows': len(df), 'columns': len(df.columns), 'duplicate_rows': int(df.duplicated().sum()), 'missing_values': int(df.isna().sum().sum())})
pd.DataFrame(profile)

,file,rows,columns,duplicate_rows,missing_values
0,customers.csv,1000,11,0,5
1,inventory.csv,2500,7,0,0
2,payments.csv,2000,5,0,0
3,products.csv,100,9,0,0
4,promotions.csv,21,6,0,0
5,sales_details.csv,4688,7,0,0
6,sales_transactions.csv,2000,9,0,943
7,stores.csv,25,7,0,0
